In [ ]:
from pathlib import Path
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS
import glob
import numpy as np

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
import warnings

warnings.filterwarnings("ignore")

In [ ]:
### cell 0 ###

benchmark_name = "kaggle-survey-2022-all-results"
pd.set_option("display.max_columns", 5000)
factor = 0.1


def load_survey_data(base_dir, file_name, rows_to_skip=1):
    file_path = os.path.join(base_dir, file_name)
    df = pd.read_csv(
        file_path, low_memory=False, encoding="ISO-8859-1", skiprows=rows_to_skip
    )
    file_name = "kaggle_survey_" + base_dir[-5:-1] + ".csv"
    files_present = glob.glob(file_name)
    if not files_present:
        df.to_csv(file_name, index=False)
    return df


directory = f"{Path(__file__).parent.parent}/kaggle/working/individual_charts/"
if not os.path.exists(directory):
    os.makedirs(directory, exist_ok=True)
directory_cell8 = f"{Path(__file__).parent.parent}/kaggle/working/individual_charts/data/"
if not os.path.exists(directory_cell8):
    os.makedirs(directory_cell8, exist_ok=True)
directory_cell8 = f"{Path(__file__).parent.parent}/kaggle/working/individual_charts/charts/"
if not os.path.exists(directory_cell8):
    os.makedirs(directory_cell8, exist_ok=True)

base_dir_2017 = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "kaggle-survey-2017"
file_name_2017 = "multipleChoiceResponses.csv"
responses_df_2017 = load_survey_data(base_dir_2017, file_name_2017, rows_to_skip=0)
responses_df_2017 = responses_df_2017.sample(frac=factor, random_state=0)

base_dir_2018 = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "kaggle-survey-2018"
file_name_2018 = "multipleChoiceResponses.csv"
responses_df_2018 = load_survey_data(base_dir_2018, file_name_2018)
responses_df_2018 = responses_df_2018.sample(frac=factor, random_state=0)

base_dir_2019 = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "kaggle-survey-2019"
file_name_2019 = "multiple_choice_responses.csv"
responses_df_2019 = load_survey_data(base_dir_2019, file_name_2019)
responses_df_2019 = responses_df_2019.sample(frac=factor, random_state=0)

base_dir_2020 = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "kaggle-survey-2020"
file_name_2020 = "kaggle_survey_2020_responses.csv"
responses_df_2020 = load_survey_data(base_dir_2020, file_name_2020)
responses_df_2020 = responses_df_2020.sample(frac=factor, random_state=0)

base_dir_2021 = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "kaggle-survey-2021"
file_name_2021 = "kaggle_survey_2021_responses.csv"
responses_df_2021 = load_survey_data(base_dir_2021, file_name_2021)
responses_df_2021 = responses_df_2021.sample(frac=factor, random_state=0)

base_dir_2022 = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "kaggle-survey-2022"
file_name_2022 = "kaggle_survey_2022_responses.csv"
responses_df_2022 = load_survey_data(base_dir_2022, file_name_2022)
responses_df_2022 = responses_df_2022.sample(frac=factor, random_state=0)

In [ ]:
### cell 1 ###

responses_df_2018_cell10 = responses_df_2018[
    responses_df_2018.columns.drop(list(responses_df_2018.filter(regex="- Text")))
]
responses_df_2019_cell10 = responses_df_2019[
    responses_df_2019.columns.drop(list(responses_df_2019.filter(regex="- Text")))
]


def replace_hyphen_with_en_dash(df):
    df.columns = df.columns.str.replace("Scikit-learn", "Scikit—learn")
    df.columns = df.columns.str.replace("peer-reviewed", "peer—reviewed")
    df.columns = df.columns.str.replace("Cloud-certification", "Cloud—certification")
    df.columns = df.columns.str.replace("U-Net, Mask R-CNN", "U—Net, Mask R—CNN")
    df.columns = df.columns.str.replace("Encoder-decoder", "Encoder—decoder")
    df.columns = df.columns.str.replace("GPT-3", "GPT—3")
    df.columns = df.columns.str.replace("gpt-3", "gpt—3")
    df.columns = df.columns.str.replace("pre-trained", "pre—trained")
    df.columns = df.columns.str.replace("What-if", "What—if")
    df.columns = df.columns.str.replace("Audit-AI", "Audit—AI")
    return df


responses_df_2022_cell10 = replace_hyphen_with_en_dash(responses_df_2022)

In [ ]:
### cell 2 ###

responses_df_2022_cell10.head(5)

In [ ]:
### cell 3 ###


def create_dataframe_of_counts_16(
    dataframe, column, rename_index, rename_column, return_percentages=False
):
    df = dataframe[column].value_counts().reset_index()
    if return_percentages == True:
        df[column] = df[column] * 100 / df[column].sum()
    df = pd.DataFrame(df)
    df = df.rename({"index": rename_index, column: rename_column}, axis="columns")
    return df


percentages_per_country_df = create_dataframe_of_counts_16(
    responses_df_2022_cell10[1:],
    "In which country do you currently reside?",
    "country",
    "% of respondents",
    return_percentages=False,
)
percentages_per_country_df.info()

In [ ]:
### cell 4 ###


def count_then_return_percent_17(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest = "In which country do you currently reside?"
responses_df_2022_cell10.rename(
    columns={
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom (UK)"
    },
    inplace=True,
)
responses_df_2022_cell10.replace(
    ["United Kingdom of Great Britain and Northern Ireland"],
    "United Kingdom (UK)",
    inplace=True,
)
responses_in_order = [
    "Other",
    "India",
    "United States of America",
    "Brazil",
    "Nigeria",
    "Pakistan",
    "Japan",
    "China",
    "Egypt",
    "Indonesia",
    "Mexico",
    "Turkey",
    "Russia",
]
responses_df_2022_cell10[question_of_interest][
    ~responses_df_2022_cell10[question_of_interest].isin(responses_in_order)
] = "Other"
percentages = count_then_return_percent_17(
    responses_df_2022_cell10, question_of_interest
).sort_index()[responses_in_order]
percentages_cell17 = percentages.sort_values(ascending=True)
orientation_for_chart = "h"
percentages_cell17.info()

In [ ]:
### cell 5 ###


def count_then_return_percent_18(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_18(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_18(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_18(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_18(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_name_alternate = "Country"
responses_df_2022_cell10.rename(
    columns={
        "United Kingdom (UK)": "United Kingdom of Great Britain and Northern Ireland"
    },
    inplace=True,
)
responses_df_2022_cell10.replace(
    ["United Kingdom (UK)"],
    "United Kingdom of Great Britain and Northern Ireland",
    inplace=True,
)
responses_df_2017[question_name_alternate].replace(
    ["United States"], "United States of America", inplace=True
)
responses_df_2017[question_name_alternate].replace(
    ["People 's Republic of China"], "China", inplace=True
)
responses_df_2017[question_name_alternate].replace(
    ["United Kingdom"],
    "United Kingdom of Great Britain and Northern Ireland",
    inplace=True,
)
responses_df_2017.rename(
    columns={question_name_alternate: "In which country do you currently reside?"},
    inplace=True,
)
question_name_alternate_cell18 = "CurrentJobTitleSelect"
responses_df_2017.rename(
    columns={
        question_name_alternate_cell18: "Select the title most similar to your current role (or most recent title if retired): - Selected Choice"
    },
    inplace=True,
)
subset_of_countries = [
    "Other",
    "India",
    "United States of America",
    "Brazil",
    "Nigeria",
    "Pakistan",
    "Japan",
    "China",
    "Egypt",
    "Indonesia",
    "Mexico",
    "Turkey",
    "Russia",
]
question_name = "In which country do you currently reside?"
responses_df_2017[question_name][
    ~responses_df_2017[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2018_cell10[question_name][
    ~responses_df_2018_cell10[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2019_cell10[question_name][
    ~responses_df_2019_cell10[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2020[question_name][
    ~responses_df_2020[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2021[question_name][
    ~responses_df_2021[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2022_cell10[question_name][
    ~responses_df_2022_cell10[question_name].isin(subset_of_countries)
] = "Other"
question_of_interest_cell18 = "In which country do you currently reside?"
title_for_x_axis = ""
country_df_combined = combine_subset_of_data_from_multiple_years_18(
    question_of_interest_cell18, title_for_x_axis, include_2017="yes"
)
country_df_combined_cell18 = country_df_combined.sort_values(
    by=["year", "percentage"], ascending=True
)
country_df_combined_cell18.rename(
    columns={
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom (UK)"
    },
    inplace=True,
)
country_df_combined_cell18.replace(
    ["United Kingdom of Great Britain and Northern Ireland"],
    "United Kingdom",
    inplace=True,
)
country_df_combined_cell18.info()

In [ ]:
### cell 6 ###


def count_then_return_percent_19(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell19 = "What is your age (# years)?"
percentages_cell19 = count_then_return_percent_19(
    responses_df_2022_cell10, question_of_interest_cell19
).sort_index()
percentages_cell19.info()

In [ ]:
### cell 7 ###


def count_then_return_percent_20(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_20(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_20(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_20(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_20(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell20 = "What is your age (# years)?"
title_for_x_axis_cell20 = ""
responses_df_2018_cell10[question_of_interest_cell20].replace(
    ["70-79", "80+"], "70+", inplace=True
)
age_df_combined = combine_subset_of_data_from_multiple_years_20(
    question_of_interest_cell20, title_for_x_axis_cell20
)
age_df_combined.info()

In [ ]:
### cell 8 ###


def count_then_return_percent_21(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell21 = "What is your gender? - Selected Choice"
responses_in_order_cell21 = [
    "Man",
    "Woman",
    "Nonbinary",
    "Prefer to self-describe",
    "Prefer not to say",
]
percentages_cell21 = (
    count_then_return_percent_21(responses_df_2022_cell10, question_of_interest_cell21)
    .sort_index()[responses_in_order_cell21]
    .iloc[::-1]
)
percentages_cell21.info()

In [ ]:
### cell 9 ###


def count_then_return_percent_22(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_22(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_22(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_22(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_22(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


responses_df_2017["GenderSelect"].replace(["Male"], "Man", inplace=True)
responses_df_2017["GenderSelect"].replace(["Female"], "Woman", inplace=True)
responses_df_2017["GenderSelect"].replace(
    ["A different identity", "Non-binary, genderqueer, or gender non-conforming"],
    "Prefer to self-describe",
    inplace=True,
)
responses_df_2017["GenderSelect"].replace(
    ["Non-binary, genderqueer, or gender non-conforming"], "Nonbinary", inplace=True
)
responses_df_2018_cell10["What is your gender? - Selected Choice"].replace(
    ["Male"], "Man", inplace=True
)
responses_df_2018_cell10["What is your gender? - Selected Choice"].replace(
    ["Female"], "Woman", inplace=True
)
responses_df_2019_cell10["What is your gender? - Selected Choice"].replace(
    ["Male"], "Man", inplace=True
)
responses_df_2019_cell10["What is your gender? - Selected Choice"].replace(
    ["Female"], "Woman", inplace=True
)
responses_df_2017["GenderSelect"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2017.columns = responses_df_2017.columns.str.replace(
    "GenderSelect", "What is your gender? - Selected Choice", regex=False
)
responses_df_2018_cell10["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2019_cell10["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2020["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2021["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2022_cell10["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
question_of_interest_cell22 = "What is your gender? - Selected Choice"
title_for_x_axis_cell22 = ""
age_df_combined_cell22 = combine_subset_of_data_from_multiple_years_22(
    question_of_interest_cell22, title_for_x_axis_cell22, include_2017="yes"
)
age_df_combined_cell22 = age_df_combined.sort_values(
    by=["year", "percentage"], ascending=True
)
age_df_combined_cell22.info()

In [ ]:
### cell 10 ###


def count_then_return_percent_23(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell23 = (
    "Are you currently a student? (high school, university, or graduate)"
)
percentages_cell23 = count_then_return_percent_23(
    responses_df_2022_cell10, question_of_interest_cell23
).sort_index()
percentages_cell23

In [ ]:
### cell 11 ###


def bar_chart_multiple_choice_24(
    response_counts, title, y_axis_title, orientation="h", num_choices=15
):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ""
    pd.DataFrame(response_counts_series).to_csv(
        f"{Path(__file__).parent.parent}/kaggle/working/individual_charts/data/"
        + title
        + ".csv",
        index=True,
    )
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)


def count_then_return_percent_24(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


USA_responses_df_2022 = responses_df_2022_cell10[
    responses_df_2022_cell10["In which country do you currently reside?"]
    == "United States of America"
]
question_of_interest_cell24 = (
    "Are you currently a student? (high school, university, or graduate)"
)
percentages_cell24 = count_then_return_percent_24(
    USA_responses_df_2022, question_of_interest_cell24
).sort_index()
title_for_chart_cell24 = "Students status for Kaggle Survey participants from the USA"
title_for_y_axis_cell24 = "% of respondents"
bar_chart_multiple_choice_24(
    response_counts=percentages_cell24,
    title=title_for_chart_cell24,
    y_axis_title=title_for_y_axis_cell24,
    orientation=orientation_for_chart,
)
India_responses_df_2022 = responses_df_2022_cell10[
    responses_df_2022_cell10["In which country do you currently reside?"] == "India"
]
question_of_interest_cell24 = (
    "Are you currently a student? (high school, university, or graduate)"
)
percentages_cell24 = count_then_return_percent_24(
    India_responses_df_2022, question_of_interest_cell24
).sort_index()
title_for_chart_cell24 = "Students status for Kaggle Survey participants from India"
title_for_y_axis_cell24 = "% of respondents"
percentages_cell24.info()

In [ ]:
### cell 12 ###


def grab_subset_of_data_25(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell25 = (
    "On which platforms have you begun or completed data science courses?"
)
online_learning_platforms_df_2022 = grab_subset_of_data_25(
    responses_df_2022_cell10, question_of_interest_cell25
)
online_learning_platforms_df_2022.columns = (
    online_learning_platforms_df_2022.columns.str.replace(
        "(direct from AWS, Azure, GCP, or similar)", "", regex=False
    )
)
online_learning_platforms_df_2022.columns = (
    online_learning_platforms_df_2022.columns.str.replace(
        "(resulting in a university degree)", "", regex=False
    )
)
online_learning_platforms_df_2022.info()

In [ ]:
### cell 13 ###


def grab_subset_of_data_26(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_26(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_26(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_26(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_26(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_26(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_26(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_26(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_26(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_26(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_26(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_26(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_26(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_26(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_26(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_26(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_cell26 = (
    "On which platforms have you begun or completed data science courses"
)
question_of_interest_alternate = (
    "On which online platforms have you begun or completed data science courses"
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_alternate, question_of_interest_cell26
)
responses_df_2018_cell10.replace(["Kaggle Learn"], "Kaggle Learn Courses", inplace=True)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Kaggle Learn", "Kaggle Learn Courses", regex=False
)
responses_df_2018_cell10.replace(["Fast.AI"], "Fast.ai", inplace=True)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Fast.AI", "Fast.ai"
)
responses_df_2018_cell10.replace(
    ["Online University Courses"],
    "University Courses (resulting in a university degree)",
    inplace=True,
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Online University Courses",
    "University Courses (resulting in a university degree)",
    regex=False,
)
responses_df_2019_cell10.replace(
    ["Kaggle Courses (i.e. Kaggle Learn)"], "Kaggle Learn Courses", inplace=True
)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    "Kaggle Courses (i.e. Kaggle Learn)", "Kaggle Learn Courses", regex=False
)
question_of_interest_cell26 = (
    "On which platforms have you begun or completed data science courses?"
)
(learning_platform_df_combined, learning_platform_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
        question_of_interest_cell26
    )
)
learning_platform_df_combined_percentages = convert_df_of_counts_to_percentages_26(
    learning_platform_df_combined, learning_platform_df_combined_counts
)
learning_platform_df_combined_percentages.columns = (
    learning_platform_df_combined_percentages.columns.str.replace(
        "(resulting in a university degree)", "", regex=False
    )
)
learning_platform_df_combined_percentages_cell26 = (
    learning_platform_df_combined_percentages.loc[
        :,
        [
            "year",
            "Coursera",
            "University Courses ",
            "Kaggle Learn Courses",
            "Udemy",
            "Udacity",
            "DataCamp",
            "edX",
            "Fast.ai",
            "None",
            "Other",
        ],
    ]
)
df = learning_platform_df_combined_percentages_cell26.melt(
    id_vars=["year"],
    value_vars=[
        "Coursera",
        "University Courses ",
        "Kaggle Learn Courses",
        "Udemy",
        "Udacity",
        "DataCamp",
        "edX",
        "Fast.ai",
    ],
)
df_cell26 = df.sort_values(by=["year", "value"], ascending=True)
df_cell26.columns = df_cell26.columns.str.replace("variable", "")
df_cell26.info()

In [ ]:
### cell 14 ###


def grab_subset_of_data_27(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell27 = "What products or platforms did you find to be most helpful when you first started studying data science?"
online_learning_platforms_df_2022_cell27 = grab_subset_of_data_27(
    responses_df_2022_cell10, question_of_interest_cell27
)
online_learning_platforms_df_2022_cell27.info()

In [ ]:
### cell 15 ###


def count_then_return_percent_28(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell28 = "What is the highest level of formal education that you have attained or plan to attain within the next 2 years?"
responses_df_2022_cell10[question_of_interest_cell28].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
responses_df_2022_cell10[question_of_interest_cell28].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2022_cell10[question_of_interest_cell28].replace(
    ["Some college/university study without earning a bachelorâ\x80\x99s degree"],
    "Some university study but without earning a degree",
    inplace=True,
)
responses_in_order_cell28 = [
    "No formal education past high school",
    "Some university study but without earning a degree",
    "Bachelor's degree",
    "Master's degree",
    "Doctoral degree",
    "Professional doctorate",
]
percentages_cell28 = count_then_return_percent_28(
    responses_df_2022_cell10, question_of_interest_cell28
).sort_index()[responses_in_order_cell28]
percentages_cell28.info()

In [ ]:
### cell 16 ###


def count_then_return_percent_29(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_29(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_29(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_29(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_29(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell29 = "What is the highest level of formal education that you have attained or plan to attain within the next 2 years?"
responses_df_2017.rename(
    columns={"FormalEducation": question_of_interest_cell29}, inplace=True
)
responses_df_2017[question_of_interest_cell29].replace(
    ["Master's degree"], "Master's degree", inplace=True
)
responses_df_2017[question_of_interest_cell29].replace(
    ["Bachelor's degree"], "Bachelor's degree", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell29].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2019_cell10[question_of_interest_cell29].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2020[question_of_interest_cell29].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2021[question_of_interest_cell29].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell29].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
responses_df_2019_cell10[question_of_interest_cell29].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
responses_df_2020[question_of_interest_cell29].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
responses_df_2021[question_of_interest_cell29].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
question_of_interest_cell29 = "What is the highest level of formal education that you have attained or plan to attain within the next 2 years?"
title_for_x_axis_cell29 = ""
degree_df_combined = combine_subset_of_data_from_multiple_years_29(
    question_of_interest_cell29, title_for_x_axis_cell29, include_2017="yes"
)
degree_df_combined_cell29 = degree_df_combined.sort_values(
    by=["year", "percentage"], ascending=True
)
subset_of_degrees = ["Bachelor's degree", "Master's degree", "Doctoral degree"]
degree_df_combined_cell29[""][
    ~degree_df_combined_cell29[""].isin(subset_of_degrees)
] = "Other"
degree_df_combined_cell29.info()

In [ ]:
### cell 17 ###


def count_then_return_percent_30(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell30 = "Have you ever published any academic research (papers, preprints, conference proceedings, etc)?"
percentages_cell30 = count_then_return_percent_30(
    responses_df_2022_cell10, question_of_interest_cell30
).sort_index()
percentages_cell30.info()

In [ ]:
### cell 18 ###


def grab_subset_of_data_31(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell31 = "Did your research make use of machine learning?"
online_learning_platforms_df_2022_cell31 = grab_subset_of_data_31(
    responses_df_2022_cell10, question_of_interest_cell31
)
online_learning_platforms_df_2022_cell31.info()

In [ ]:
### cell 19 ###


def count_then_return_percent_32(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell32 = (
    "For how many years have you been writing code and/or programming?"
)
responses_in_order_cell32 = [
    "I have never written code",
    "< 1 years",
    "1-3 years",
    "3-5 years",
    "5-10 years",
    "10-20 years",
    "20+ years",
]
percentages_cell32 = count_then_return_percent_32(
    responses_df_2022_cell10, question_of_interest_cell32
).sort_index()[responses_in_order_cell32]
percentages_cell32.info()

In [ ]:
### cell 20 ###


def count_then_return_percent_33(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_33(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_33(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_33(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_33(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell33 = (
    "For how many years have you been writing code and/or programming?"
)
responses_df_2018_cell10.rename(
    columns={
        "How long have you been writing code to analyze data?": question_of_interest_cell33
    },
    inplace=True,
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["< 1 year"], "< 1 years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    [
        "I have never written code but I want to learn",
        "I have never written code and I do not want to learn",
    ],
    "I have never written code",
    inplace=True,
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["1-2 years"], "1-3 years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["20-30 years"], "20+ years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["30-40 years"], "20+ years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["40+ years"], "20+ years", inplace=True
)
responses_df_2019_cell10.rename(
    columns={
        "How long have you been writing code to analyze data (at work or at school)?": question_of_interest_cell33
    },
    inplace=True,
)
responses_df_2019_cell10[question_of_interest_cell33].replace(
    ["1-2 years"], "1-3 years", inplace=True
)
responses_df_2020[question_of_interest_cell33].replace(
    ["1-2 years"], "1-3 years", inplace=True
)
title_for_x_axis_cell33 = ""
programming_df_combined = combine_subset_of_data_from_multiple_years_33(
    question_of_interest_cell33, title_for_x_axis_cell33
).sort_values(by=["year", "percentage"], ascending=True)
programming_df_combined.info()

In [ ]:
### cell 21 ###


def grab_subset_of_data_34(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell34 = (
    "What programming languages do you use on a regular basis?"
)
online_learning_platforms_df_2022_cell34 = grab_subset_of_data_34(
    responses_df_2022_cell10, question_of_interest_cell34
)
online_learning_platforms_df_2022_cell34.info()

In [ ]:
### cell 22 ###


def grab_subset_of_data_35(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_35(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_35(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_35(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_35(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_35(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_35(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_35(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_35(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_35(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_35(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_35(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_35(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_35(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_35(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_35(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_cell35 = (
    "What programming languages do you use on a regular basis?"
)
(language_df_combined, language_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest_cell35
    )
)
language_df_combined_percentages = convert_df_of_counts_to_percentages_35(
    language_df_combined, language_df_combined_counts
)
language_df_combined_percentages_cell35 = language_df_combined_percentages.loc[
    :, ["year", "Python", "SQL", "C++", "C", "R", "Java", "Javascript", "Other"]
]
df_cell35 = language_df_combined_percentages_cell35.melt(
    id_vars=["year"],
    value_vars=["Python", "SQL", "C++", "C", "R", "Java", "Javascript", "Other"],
)
df_cell35 = df.sort_values(by=["year", "value"], ascending=True)
df_cell35 = df.rename(columns={"variable": ""})
df_cell35.info()

In [ ]:
### cell 23 ###


def grab_subset_of_data_36(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell36 = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
ide_df_2022 = grab_subset_of_data_36(
    responses_df_2022_cell10, question_of_interest_cell36
)
ide_df_2022.info()

In [ ]:
### cell 24 ###


def grab_subset_of_data_37(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell37 = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
ide_df_2022_cell37 = grab_subset_of_data_37(
    responses_df_2022_cell10, question_of_interest_cell37
)
columns_to_combine = ["Jupyter Notebook", "JupyterLab "]
ide_df_2022_cell37["Jupyter/JupyterLab"] = (
    ide_df_2022_cell37[columns_to_combine].notna().any(axis="columns")
)
ide_df_2022_cell37 = ide_df_2022_cell37.drop(columns=columns_to_combine)
ide_df_2022_cell37["Jupyter/JupyterLab"].replace(
    [True], "Jupyter/JupyterLab", inplace=True
)
ide_df_2022_cell37["Jupyter/JupyterLab"].replace([False], np.nan, inplace=True)
columns_to_combine_cell37 = ["Visual Studio Code (VSCode) ", "Visual Studio "]
ide_df_2022_cell37["Visual Studio / Visual Studio Code (VSCode)"] = (
    ide_df_2022_cell37[columns_to_combine_cell37].notna().any(axis="columns")
)
ide_df_2022_cell37 = ide_df_2022_cell37.drop(columns=columns_to_combine_cell37)
ide_df_2022_cell37["Visual Studio / Visual Studio Code (VSCode)"].replace(
    [True], "Visual Studio / Visual Studio Code (VSCode)", inplace=True
)
ide_df_2022_cell37["Visual Studio / Visual Studio Code (VSCode)"].replace(
    [False], np.nan, inplace=True
)
ide_df_2022_cell37.info()

In [ ]:
### cell 25 ###


def grab_subset_of_data_38(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_38(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_38(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_38(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_38(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_38(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_38(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_38(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_38(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_38(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_38(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_38(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_38(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_38(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_38(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_38(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


correct_phrasing = "Jupyter Notebook / JupyterLab"
incorrect_phrasing = "Jupyter/IPython"
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    incorrect_phrasing, correct_phrasing, regex=False
)
question_of_interest_cell38 = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
alternate_phrasing = "Which of the following integrated development environments (IDE's) have you used at work or school in the last 5 years?"
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    alternate_phrasing, question_of_interest_cell38, regex=False
)
x_axis_title = "Percentage of respondents"
(ide_df_combined, ide_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
        question_of_interest_cell38
    )
)
ide_df_combined_2 = ide_df_combined.copy()
columns_to_combine_cell38 = [
    "Jupyter Notebook",
    "JupyterLab ",
    "Jupyter (JupyterLab, Jupyter Notebooks, etc) ",
    "Jupyter Notebook / JupyterLab",
]
ide_df_combined_2["Jupyter Notebook / JupyterLab_"] = (
    ide_df_combined_2[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2.drop(columns=columns_to_combine_cell38)
ide_df_combined_2_cell38["Jupyter Notebook / JupyterLab_"].replace(
    [True], "Jupyter Notebook / JupyterLab", inplace=True
)
ide_df_combined_2_cell38["Jupyter Notebook / JupyterLab_"].replace(
    [False], np.nan, inplace=True
)
ide_df_combined_2_cell38.columns = ide_df_combined_2_cell38.columns.str.replace(
    "Jupyter Notebook / JupyterLab_", "Jupyter Notebook / JupyterLab", regex=False
)
columns_to_combine_cell38 = ["MATLAB", "MATLAB "]
ide_df_combined_2_cell38["MATLAB_"] = (
    ide_df_combined_2_cell38[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2_cell38.drop(
    columns=columns_to_combine_cell38
)
ide_df_combined_2_cell38["MATLAB_"].replace([True], "MATLAB", inplace=True)
ide_df_combined_2_cell38["MATLAB_"].replace([False], np.nan, inplace=True)
ide_df_combined_2_cell38.columns = ide_df_combined_2_cell38.columns.str.replace(
    "MATLAB_", "MATLAB", regex=False
)
columns_to_combine_cell38 = ["RStudio", "RStudio "]
ide_df_combined_2_cell38["RStudio_"] = (
    ide_df_combined_2_cell38[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2_cell38.drop(
    columns=columns_to_combine_cell38
)
ide_df_combined_2_cell38["RStudio_"].replace([True], "RStudio", inplace=True)
ide_df_combined_2_cell38["RStudio_"].replace([False], np.nan, inplace=True)
ide_df_combined_2_cell38.columns = ide_df_combined_2_cell38.columns.str.replace(
    "RStudio_", "RStudio", regex=False
)
columns_to_combine_cell38 = [
    "Visual Studio Code",
    "Visual Studio",
    "Visual Studio / Visual Studio Code ",
    "Visual Studio ",
    "Visual Studio Code (VSCode) ",
    "Click to write Choice 13",
]
ide_df_combined_2_cell38["Visual Studio / Visual Studio Code (VSCode)"] = (
    ide_df_combined_2_cell38[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2_cell38.drop(
    columns=columns_to_combine_cell38
)
ide_df_combined_2_cell38["Visual Studio / Visual Studio Code (VSCode)"].replace(
    [True], "Visual Studio / Visual Studio Code (VSCode)", inplace=True
)
ide_df_combined_2_cell38["Visual Studio / Visual Studio Code (VSCode)"].replace(
    [False], np.nan, inplace=True
)
columns_to_combine_cell38 = ["PyCharm ", "PyCharm"]
ide_df_combined_2_cell38["PyCharm_"] = (
    ide_df_combined_2_cell38[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2_cell38.drop(
    columns=columns_to_combine_cell38
)
ide_df_combined_2_cell38["PyCharm_"].replace([True], "PyCharm", inplace=True)
ide_df_combined_2_cell38["PyCharm_"].replace([False], np.nan, inplace=True)
ide_df_combined_2_cell38.columns = ide_df_combined_2_cell38.columns.str.replace(
    "PyCharm_", "PyCharm", regex=False
)
ide_df_combined_counts_2 = (
    ide_df_combined_2_cell38.groupby("year").count().reset_index()
)
ide_df_combined_percentages = convert_df_of_counts_to_percentages_38(
    ide_df_combined_2_cell38, ide_df_combined_counts_2
)
ide_df_combined_percentages_cell38 = ide_df_combined_percentages.loc[
    :,
    [
        "year",
        "Jupyter Notebook / JupyterLab",
        "Visual Studio / Visual Studio Code (VSCode)",
        "RStudio",
        "PyCharm",
        "MATLAB",
    ],
]
df_cell38 = ide_df_combined_percentages_cell38.melt(
    id_vars=["year"],
    value_vars=[
        "Jupyter Notebook / JupyterLab",
        "Visual Studio / Visual Studio Code (VSCode)",
        "RStudio",
        "PyCharm",
        "MATLAB",
    ],
)
df_cell38 = df.sort_values(by=["year", "value"], ascending=True)
df_cell38 = df.rename(columns={"variable": ""})
df_cell38.info()

In [ ]:
### cell 26 ###


def grab_subset_of_data_39(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell39 = (
    "Do you use any of the following hosted notebook products?"
)
notebooks_df_2022 = grab_subset_of_data_39(
    responses_df_2022_cell10, question_of_interest_cell39
)
notebooks_df_2022.info()

In [ ]:
### cell 27 ###


def grab_subset_of_data_40(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_40(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_40(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_40(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_40(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_40(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_40(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_40(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_40(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_40(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_40(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_40(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_40(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_40(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_40(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_40(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_original = (
    "Which of the following hosted notebook products do you use on a regular basis?"
)
question_of_interest_new = "Do you use any of the following hosted notebook products?"
responses_df_2021.columns = responses_df_2021.columns.str.replace(
    question_of_interest_original, question_of_interest_new, regex=False
)
question_of_interest_original_cell40 = (
    "Which of the following hosted notebook products do you use on a regular basis?"
)
question_of_interest_new_cell40 = (
    "Do you use any of the following hosted notebook products?"
)
responses_df_2020.columns = responses_df_2020.columns.str.replace(
    question_of_interest_original_cell40, question_of_interest_new_cell40, regex=False
)
colab_text_to_replace = "Google Colab "
colab_new_text = "Colab Notebooks"
colab_answer = "Which of the following hosted notebook products do you use on a regular basis?  (Select all that apply) - Selected Choice -  Google Colab "
kaggle_text_to_replace = "Kaggle Notebooks (Kernels) "
kaggle_new_text = "Kaggle Notebooks"
kaggle_answer = "Which of the following hosted notebook products do you use on a regular basis?  (Select all that apply) - Selected Choice -  Kaggle Notebooks (Kernels) "
responses_df_2019_cell10[colab_answer] = responses_df_2019_cell10[
    colab_answer
].str.replace(colab_text_to_replace, colab_new_text, regex=False)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    colab_text_to_replace, colab_new_text, regex=False
)
responses_df_2019_cell10[kaggle_answer] = responses_df_2019_cell10[
    kaggle_answer
].str.replace(kaggle_text_to_replace, kaggle_new_text, regex=False)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    kaggle_text_to_replace, kaggle_new_text, regex=False
)
question_of_interest_original_cell40 = (
    "Which of the following hosted notebook products do you use on a regular basis?"
)
question_of_interest_new_cell40 = (
    "Do you use any of the following hosted notebook products?"
)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    question_of_interest_original_cell40, question_of_interest_new_cell40, regex=False
)
colab_text_to_replace_cell40 = "Google Colab"
colab_new_text_cell40 = "Colab Notebooks"
colab_answer_cell40 = "Which of the following hosted notebooks have you used at work or school in the last 5 years? (Select all that apply) - Selected Choice - Google Colab"
kaggle_text_to_replace_cell40 = "Kaggle Kernels"
kaggle_new_text_cell40 = "Kaggle Notebooks"
kaggle_answer_cell40 = "Which of the following hosted notebooks have you used at work or school in the last 5 years? (Select all that apply) - Selected Choice - Kaggle Kernels"
responses_df_2018_cell10[colab_answer_cell40] = responses_df_2018_cell10[
    colab_answer_cell40
].str.replace(colab_text_to_replace_cell40, colab_new_text_cell40, regex=False)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    colab_text_to_replace_cell40, colab_new_text_cell40, regex=False
)
responses_df_2018_cell10[kaggle_answer_cell40] = responses_df_2018_cell10[
    kaggle_answer_cell40
].str.replace(kaggle_text_to_replace_cell40, kaggle_new_text_cell40, regex=False)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    kaggle_text_to_replace_cell40, kaggle_new_text_cell40, regex=False
)
question_of_interest_original_cell40 = "Which of the following hosted notebooks have you used at work or school in the last 5 years?"
question_of_interest_new_cell40 = (
    "Do you use any of the following hosted notebook products?"
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_original_cell40, question_of_interest_new_cell40, regex=False
)
question_of_interest_cell40 = (
    "Do you use any of the following hosted notebook products?"
)
(notebooks_df_combined, notebooks_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest_cell40
    )
)
notebooks_df_combined_percentages = convert_df_of_counts_to_percentages_40(
    notebooks_df_combined, notebooks_df_combined_counts
)
notebooks_df_combined_percentages_cell40 = notebooks_df_combined_percentages.loc[
    :, ["year", "None", "Kaggle Notebooks", "Colab Notebooks"]
]
df_cell40 = notebooks_df_combined_percentages_cell40.melt(
    id_vars=["year"], value_vars=["None", "Kaggle Notebooks", "Colab Notebooks"]
)
df_cell40 = df.rename(columns={"variable": ""})
df_cell40.info()

In [ ]:
### cell 28 ###


def grab_subset_of_data_41(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell41 = (
    "Do you use any of the following data visualization libraries on a regular basis?"
)
notebooks_df_2022_cell41 = grab_subset_of_data_41(
    responses_df_2022_cell10, question_of_interest_cell41
)
notebooks_df_2022_cell41.info()

In [ ]:
### cell 29 ###


def count_then_return_percent_42(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell42 = (
    "For how many years have you used machine learning methods?"
)
responses_in_order_cell42 = [
    "I do not use machine learning methods",
    "Under 1 year",
    "1-2 years",
    "2-3 years",
    "3-4 years",
    "4-5 years",
    "5-10 years",
    "10-20 years",
]
percentages_cell42 = count_then_return_percent_42(
    responses_df_2022_cell10, question_of_interest_cell42
).sort_index()[responses_in_order_cell42]
percentages_cell42.info()

In [ ]:
### cell 30 ###


def count_then_return_percent_43(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_43(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_43(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_43(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_43(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell43 = (
    "For how many years have you used machine learning methods?"
)
responses_df_2018_cell10.rename(
    columns={
        "For how many years have you used machine learning methods (at work or in school)?": question_of_interest_cell43
    },
    inplace=True,
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["< 1 year"], "Under 1 year", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["10-15 years"], "10-20 years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["20+ years"], "10-20 years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["I have never studied machine learning but plan to learn in the future"],
    "I do not use machine learning methods",
    inplace=True,
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["I have never studied machine learning and I do not plan to"],
    "I do not use machine learning methods",
    inplace=True,
)
responses_df_2019_cell10[question_of_interest_cell43].replace(
    ["< 1 years"], "Under 1 year", inplace=True
)
responses_df_2019_cell10[question_of_interest_cell43].replace(
    ["10-15 years"], "10-20 years", inplace=True
)
responses_df_2019_cell10[question_of_interest_cell43].replace(
    ["20+ years"], "20 or more years", inplace=True
)
title_for_x_axis_cell43 = ""
ml_exp_df_combined = combine_subset_of_data_from_multiple_years_43(
    question_of_interest_cell43, title_for_x_axis_cell43
).sort_values(by=["year", "percentage"], ascending=True)
ml_exp_df_combined.info()

In [ ]:
### cell 31 ###


def grab_subset_of_data_44(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell44 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
ml_frameworks_df_2022 = grab_subset_of_data_44(
    responses_df_2022_cell10, question_of_interest_cell44
)
ml_frameworks_df_2022.info()

In [ ]:
### cell 32 ###


def grab_subset_of_data_45(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


ml_frameworks_df_2022_cell45 = grab_subset_of_data_45(
    responses_df_2022_cell10, question_of_interest_cell44
)
columns_to_combine_cell45 = ["TensorFlow ", "Keras "]
ml_frameworks_df_2022_cell45["TensorFlow/Keras"] = (
    ml_frameworks_df_2022_cell45[columns_to_combine_cell45].notna().any(axis="columns")
)
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45.drop(
    columns=columns_to_combine_cell45
)
ml_frameworks_df_2022_cell45["TensorFlow/Keras"].replace(
    [True], "TensorFlow/Keras", inplace=True
)
ml_frameworks_df_2022_cell45["TensorFlow/Keras"].replace([False], np.nan, inplace=True)
columns_to_combine_cell45 = ["PyTorch ", "PyTorch Lightning ", "Fast.ai "]
ml_frameworks_df_2022_cell45["PyTorch/PyTorch Lightning/Fast.ai"] = (
    ml_frameworks_df_2022_cell45[columns_to_combine_cell45].notna().any(axis="columns")
)
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45.drop(
    columns=columns_to_combine_cell45
)
ml_frameworks_df_2022_cell45["PyTorch/PyTorch Lightning/Fast.ai"].replace(
    [True], "PyTorch/PyTorch Lightning/Fast.ai", inplace=True
)
ml_frameworks_df_2022_cell45["PyTorch/PyTorch Lightning/Fast.ai"].replace(
    [False], np.nan, inplace=True
)
columns_to_combine_cell45 = ["Xgboost ", "LightGBM ", "CatBoost "]
ml_frameworks_df_2022_cell45["Xgboost/LightGBM/CatBoost"] = (
    ml_frameworks_df_2022_cell45[columns_to_combine_cell45].notna().any(axis="columns")
)
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45.drop(
    columns=columns_to_combine_cell45
)
ml_frameworks_df_2022_cell45["Xgboost/LightGBM/CatBoost"].replace(
    [True], "Xgboost/LightGBM/CatBoost", inplace=True
)
ml_frameworks_df_2022_cell45["Xgboost/LightGBM/CatBoost"].replace(
    [False], np.nan, inplace=True
)
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45.loc[
    :,
    [
        "Scikit—learn ",
        "TensorFlow/Keras",
        "PyTorch/PyTorch Lightning/Fast.ai",
        "Xgboost/LightGBM/CatBoost",
    ],
]
ml_frameworks_df_2022_cell45.info()

In [ ]:
### cell 33 ###


def grab_subset_of_data_46(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_46(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_46(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_46(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_46(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_46(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_46(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_46(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_46(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_46(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_46(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_46(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_46(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_46(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_46(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_46(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_old = (
    "What machine learning frameworks have you used in the past 5 years?"
)
question_of_interest_new_cell46 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_old, question_of_interest_new_cell46, regex=False
)
question_of_interest_cell46 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
(ml_df_combined, ml_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(
        question_of_interest_cell46
    )
)
columns_to_combine_cell46 = ["TensorFlow", "TensorFlow "]
ml_df_combined["TensorFlow_"] = (
    ml_df_combined[columns_to_combine_cell46].notna().any(axis="columns")
)
ml_df_combined_cell46 = ml_df_combined.drop(columns=columns_to_combine_cell46)
ml_df_combined_cell46["TensorFlow_"].replace([True], "TensorFlow", inplace=True)
ml_df_combined_cell46["TensorFlow_"].replace([False], np.nan, inplace=True)
ml_df_combined_cell46.columns = ml_df_combined_cell46.columns.str.replace(
    "TensorFlow_", "TensorFlow ", regex=False
)
columns_to_combine_cell46 = ["Keras", "Keras "]
ml_df_combined_cell46["Keras_"] = (
    ml_df_combined_cell46[columns_to_combine_cell46].notna().any(axis="columns")
)
ml_df_combined_cell46 = ml_df_combined_cell46.drop(columns=columns_to_combine_cell46)
ml_df_combined_cell46["Keras_"].replace([True], "Keras", inplace=True)
ml_df_combined_cell46["Keras_"].replace([False], np.nan, inplace=True)
ml_df_combined_cell46.columns = ml_df_combined_cell46.columns.str.replace(
    "Keras_", "Keras ", regex=False
)
columns_to_combine_cell46 = ["PyTorch", "PyTorch "]
ml_df_combined_cell46["PyTorch_"] = (
    ml_df_combined_cell46[columns_to_combine_cell46].notna().any(axis="columns")
)
ml_df_combined_cell46 = ml_df_combined_cell46.drop(columns=columns_to_combine_cell46)
ml_df_combined_cell46["PyTorch_"].replace([True], "PyTorch", inplace=True)
ml_df_combined_cell46["PyTorch_"].replace([False], np.nan, inplace=True)
ml_df_combined_cell46.columns = ml_df_combined_cell46.columns.str.replace(
    "PyTorch_", "PyTorch ", regex=False
)
columns_to_combine_cell46 = ["Scikit—learn ", "Learn", "learn "]
ml_df_combined_cell46["Scikit—learn_"] = (
    ml_df_combined_cell46[columns_to_combine_cell46].notna().any(axis="columns")
)
ml_df_combined_cell46 = ml_df_combined_cell46.drop(columns=columns_to_combine_cell46)
ml_df_combined_cell46["Scikit—learn_"].replace([True], "Scikit—learn", inplace=True)
ml_df_combined_cell46["Scikit—learn_"].replace([False], np.nan, inplace=True)
ml_df_combined_cell46.columns = ml_df_combined_cell46.columns.str.replace(
    "Scikit—learn_", "Scikit-learn ", regex=False
)
ml_df_combined_counts_2 = ml_df_combined_cell46.groupby("year").count().reset_index()
ml_df_combined_percentages = convert_df_of_counts_to_percentages_46(
    ml_df_combined_cell46, ml_df_combined_counts_2
)
ml_df_combined_percentages_cell46 = ml_df_combined_percentages.loc[
    :, ["year", "Scikit-learn ", "TensorFlow ", "Keras ", "PyTorch ", "None", "Other"]
]
df_cell46 = ml_df_combined_percentages_cell46.melt(
    id_vars=["year"],
    value_vars=["Scikit-learn ", "TensorFlow ", "Keras ", "PyTorch ", "None", "Other"],
)
df_cell46 = df.sort_values(by=["year", "value"], ascending=True)
df_cell46 = df.rename(columns={"variable": ""})
df_cell46.info()

In [ ]:
### cell 34 ###


def grab_subset_of_data_47(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_47(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_47(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_47(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_47(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_47(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_47(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_47(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_47(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_47(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_47(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_47(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_47(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_47(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_47(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_47(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_old_cell47 = (
    "What machine learning frameworks have you used in the past 5 years?"
)
question_of_interest_new_cell47 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_old_cell47, question_of_interest_new_cell47, regex=False
)
question_of_interest_cell47 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
(ml_df_combined_cell47, ml_df_combined_counts_cell47) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
        question_of_interest_cell47
    )
)
ml_df_combined_2 = ml_df_combined_cell47.copy()
columns_to_combine_cell47 = ["TensorFlow", "TensorFlow ", "Keras", "Keras "]
ml_df_combined_2["TensorFlow/Keras"] = (
    ml_df_combined_2[columns_to_combine_cell47].notna().any(axis="columns")
)
ml_df_combined_2_cell47 = ml_df_combined_2.drop(columns=columns_to_combine_cell47)
ml_df_combined_2_cell47["TensorFlow/Keras"].replace(
    [True], "TensorFlow/Keras", inplace=True
)
ml_df_combined_2_cell47["TensorFlow/Keras"].replace([False], np.nan, inplace=True)
columns_to_combine_cell47 = [
    "PyTorch",
    "PyTorch ",
    "PyTorch Lightning ",
    "Fast.ai ",
    "Fastai",
]
ml_df_combined_2_cell47["PyTorch/Lightning/Fast.ai"] = (
    ml_df_combined_2_cell47[columns_to_combine_cell47].notna().any(axis="columns")
)
ml_df_combined_2_cell47 = ml_df_combined_2_cell47.drop(
    columns=columns_to_combine_cell47
)
ml_df_combined_2_cell47["PyTorch/Lightning/Fast.ai"].replace(
    [True], "PyTorch/PyTorch Lightning/Fast.ai", inplace=True
)
ml_df_combined_2_cell47["PyTorch/Lightning/Fast.ai"].replace(
    [False], np.nan, inplace=True
)
columns_to_combine_cell47 = [
    "Xgboost",
    "Xgboost ",
    "lightgbm",
    "LightGBM ",
    "catboost",
    "CatBoost ",
]
ml_df_combined_2_cell47["Xgboost/LightGBM/CatBoost"] = (
    ml_df_combined_2_cell47[columns_to_combine_cell47].notna().any(axis="columns")
)
ml_df_combined_2_cell47 = ml_df_combined_2_cell47.drop(
    columns=columns_to_combine_cell47
)
ml_df_combined_2_cell47["Xgboost/LightGBM/CatBoost"].replace(
    [True], "Xgboost/LightGBM/CatBoost", inplace=True
)
ml_df_combined_2_cell47["Xgboost/LightGBM/CatBoost"].replace(
    [False], np.nan, inplace=True
)
columns_to_combine_cell47 = ["Scikit—learn ", "Scikit—learn ", "learn ", "Learn"]
ml_df_combined_2_cell47["Scikit-learn_"] = (
    ml_df_combined_2_cell47[columns_to_combine_cell47].notna().any(axis="columns")
)
ml_df_combined_2_cell47 = ml_df_combined_2_cell47.drop(
    columns=columns_to_combine_cell47
)
ml_df_combined_2_cell47["Scikit-learn_"].replace([True], "Scikit-learn_", inplace=True)
ml_df_combined_2_cell47["Scikit-learn_"].replace([False], np.nan, inplace=True)
ml_df_combined_2_cell47.columns = ml_df_combined_2_cell47.columns.str.replace(
    "Scikit-learn_", "Scikit-learn", regex=False
)
ml_df_combined_counts_2_cell47 = (
    ml_df_combined_2_cell47.groupby("year").count().reset_index()
)
ml_df_combined_percentages_cell47 = convert_df_of_counts_to_percentages_47(
    ml_df_combined_2_cell47, ml_df_combined_counts_2_cell47
)
ml_df_combined_percentages_cell47 = ml_df_combined_percentages_cell47.loc[
    :,
    [
        "year",
        "Scikit-learn",
        "TensorFlow/Keras",
        "PyTorch/Lightning/Fast.ai",
        "Xgboost/LightGBM/CatBoost",
    ],
]
df_cell47 = ml_df_combined_percentages_cell47.melt(
    id_vars=["year"],
    value_vars=[
        "Scikit-learn",
        "Xgboost/LightGBM/CatBoost",
        "TensorFlow/Keras",
        "PyTorch/Lightning/Fast.ai",
    ],
)
df_cell47 = df.sort_values(by=["year", "value"], ascending=True)
df_cell47 = df.rename(columns={"variable": ""})
df_cell47.info()

In [ ]:
### cell 35 ###


def grab_subset_of_data_48(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell48 = (
    "Which of the following ML algorithms do you use on a regular basis?"
)
ml_algos_df_2022 = grab_subset_of_data_48(
    responses_df_2022_cell10, question_of_interest_cell48
)
ml_algos_df_2022.info()

In [ ]:
### cell 36 ###


def grab_subset_of_data_49(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell49 = (
    "Which categories of computer vision methods do you use on a regular basis?"
)
cv_df_2022 = grab_subset_of_data_49(
    responses_df_2022_cell10, question_of_interest_cell49
)
cv_df_2022.info()

In [ ]:
### cell 37 ###


def grab_subset_of_data_50(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell50 = "Which of the following natural language processing (NLP) methods do you use on a regular basis?"
nlp_df_2022 = grab_subset_of_data_50(
    responses_df_2022_cell10, question_of_interest_cell50
)
nlp_df_2022.info()

In [ ]:
### cell 38 ###


def grab_subset_of_data_51(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell51 = "Which of the following natural language processing (NLP) methods do you use on a regular basis?"
nlp_df_2022_cell51 = grab_subset_of_data_51(
    responses_df_2022_cell10, question_of_interest_cell51
)
counts_2022 = (
    nlp_df_2022_cell51[nlp_df_2022_cell51.columns[:]]
    .count()
    .sort_values(ascending=True)
)
percentages_2022 = counts_2022 * 100 / len(nlp_df_2022_cell51)
transformers = "Transformer language models (GPT—3, BERT, XLnet, etc)"
percentages_2022_cell51 = percentages_2022.rename({transformers: "Transformers"})[
    ["Transformers"]
]
nlp_df_2021 = grab_subset_of_data_51(responses_df_2021, question_of_interest_cell51)
counts_2021 = nlp_df_2021[nlp_df_2021.columns[:]].count().sort_values(ascending=True)
percentages_2021 = counts_2021 * 100 / len(nlp_df_2021)
transformers_cell51 = "3, BERT, XLnet, etc)"
percentages_2021_cell51 = percentages_2021.rename(
    {transformers_cell51: "Transformers"}
)[["Transformers"]]
nlp_df_2020 = grab_subset_of_data_51(responses_df_2020, question_of_interest_cell51)
counts_2020 = nlp_df_2020[nlp_df_2020.columns[:]].count().sort_values(ascending=True)
percentages_2020 = counts_2020 * 100 / len(nlp_df_2020)
transformers_cell51 = "3, BERT, XLnet, etc)"
percentages_2020_cell51 = percentages_2020.rename(
    {transformers_cell51: "Transformers"}
)[["Transformers"]]
nlp_df_2019 = grab_subset_of_data_51(
    responses_df_2019_cell10, question_of_interest_cell51
)
counts_2019 = nlp_df_2019[nlp_df_2019.columns[:]].count().sort_values(ascending=True)
percentages_2019 = counts_2019 * 100 / len(nlp_df_2019)
transformers_cell51 = "2, BERT, XLnet, etc)"
percentages_2019_cell51 = percentages_2019.rename(
    {transformers_cell51: "Transformers"}
)[["Transformers"]]
(
    percentages_2019_cell51.info(),
    percentages_2020_cell51.info(),
    percentages_2021_cell51.info(),
    percentages_2022_cell51.info(),
)

In [ ]:
### cell 39 ###


def grab_subset_of_data_52(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell52 = (
    "Do you download pre—trained model weights from any of the following services?"
)
models_df_2022 = grab_subset_of_data_52(
    responses_df_2022_cell10, question_of_interest_cell52
)
models_df_2022.info()

In [ ]:
### cell 40 ###


def bar_chart_multiple_choice_53(
    response_counts, title, y_axis_title, orientation="h", num_choices=15
):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ""
    pd.DataFrame(response_counts_series).to_csv(
        f"{Path(__file__).parent.parent}/kaggle/working/individual_charts/data/"
        + title
        + ".csv",
        index=True,
    )
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)


def count_then_return_percent_53(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell53 = "Which of the following ML model hubs/repositories do you use most often? - Selected Choice"
responses_in_order_cell53 = [
    " Kaggle datasets ",
    " Huggingface Models ",
    "  TensorFlow Hub ",
    " PyTorch Hub ",
    " Timm ",
    " NVIDIA NGC models  ",
    " ONNX models ",
    " Jumpstart ",
]
percentages_cell53 = (
    count_then_return_percent_53(responses_df_2022_cell10, question_of_interest_cell53)
    .sort_index()[responses_in_order_cell53]
    .iloc[::-1]
)
title_for_chart_cell53 = (
    "Favorite ML model repository (for users of multiple repositories)"
)
title_for_y_axis_cell53 = "% of respondents"
orientation_for_chart_cell53 = "h"
bar_chart_multiple_choice_53(
    response_counts=percentages_cell53,
    title=title_for_chart_cell53,
    y_axis_title=title_for_y_axis_cell53,
    orientation=orientation_for_chart_cell53,
    num_choices=9,
)

In [ ]:
### cell 41 ###


def count_then_return_percent_54(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell54 = "Select the title most similar to your current role (or most recent title if retired): - Selected Choice"
percentages_cell54 = count_then_return_percent_54(
    responses_df_2022_cell10, question_of_interest_cell54
).sort_index()
percentages_cell54 = percentages.sort_values(ascending=True)
percentages_cell54.info()

In [ ]:
### cell 42 ###


def count_then_return_percent_55(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_55(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_55(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_55(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_55(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell55 = "Select the title most similar to your current role (or most recent title if retired): - Selected Choice"
job_df_combined = combine_subset_of_data_from_multiple_years_55(
    question_of_interest_cell55, title_for_x_axis_cell43
)
job_df_combined.replace(
    ["Data Analyst (Business, Marketing, Financial, Quantitative, etc)"],
    "Data Analyst",
    inplace=True,
)
job_df_combined_cell55 = job_df_combined.rename(
    {"Data Analyst (Business, Marketing, Financial, Quantitative, etc)": "Data Analyst"}
)
job_df_combined_cell55 = job_df_combined[
    job_df_combined.index.isin(
        [
            "Data Analyst",
            "Data Engineer",
            "Data Scientist",
            "Research Scientist",
            "Software Engineer",
        ]
    )
]
job_df_combined_cell55.info()

In [ ]:
### cell 43 ###


def count_then_return_percent_56(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell56 = "In what industry is your current employer/contract (or your most recent employer if retired)? - Selected Choice"
percentages_cell56 = count_then_return_percent_56(
    responses_df_2022_cell10, question_of_interest_cell56
).sort_index()
percentages_cell56 = percentages.sort_values(ascending=True)
percentages_cell56.info()

In [ ]:
### cell 44 ###


def count_then_return_percent_57(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell57 = "What is the size of the company where you are employed?"
responses_in_order_cell57 = [
    "0-49 employees",
    "50-249 employees",
    "250-999 employees",
    "1000-9,999 employees",
    "10,000 or more employees",
]
percentages_cell57 = count_then_return_percent_57(
    responses_df_2022_cell10, question_of_interest_cell57
).sort_index()[responses_in_order_cell57]
percentages_cell57.info()

In [ ]:
### cell 45 ###


def count_then_return_percent_58(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell58 = "Approximately how many individuals are responsible for data science workloads at your place of business?"
responses_in_order_cell58 = ["0", "1-2", "3-4", "5-9", "10-14", "15-19", "20+"]
percentages_cell58 = count_then_return_percent_58(
    responses_df_2022_cell10, question_of_interest_cell58
).sort_index()[responses_in_order_cell58]
percentages_cell58.info()

In [ ]:
### cell 46 ###


def count_then_return_percent_59(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell59 = "Does your current employer incorporate machine learning methods into their business?"
responses_in_order_cell59 = [
    "No (we do not use ML methods)",
    "I do not know",
    "We are exploring ML methods (and may one day put a model into production)",
    "We recently started using ML methods (i.e., models in production for less than 2 years)",
    "We have well established ML methods (i.e., models in production for more than 2 years)",
]
percentages_cell59 = count_then_return_percent_59(
    responses_df_2022_cell10, question_of_interest_cell59
).sort_index()[responses_in_order_cell59]
percentages_cell59.info()

In [ ]:
### cell 47 ###


def count_then_return_percent_60(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_60(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_60(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_60(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_60(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell60 = "Does your current employer incorporate machine learning methods into their business?"
title_for_x_axis_cell60 = ""
maturity_df_combined = combine_subset_of_data_from_multiple_years_60(
    question_of_interest_cell60, title_for_x_axis_cell60
).sort_values(by=["year", "percentage"], ascending=True)
maturity_df_combined.info()

In [ ]:
### cell 48 ###


def grab_subset_of_data_61(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell61 = (
    "Select any activities that make up an important part of your role at work"
)
models_df_2022_cell61 = grab_subset_of_data_61(
    responses_df_2022_cell10, question_of_interest_cell61
)
models_df_2022_cell61.info()

In [ ]:
### cell 49 ###


def count_then_return_percent_62(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell62 = (
    "What is your current yearly compensation (approximate $USD)?"
)
responses_in_order_cell62 = [
    "$0-999",
    "1,000-1,999",
    "2,000-2,999",
    "3,000-3,999",
    "4,000-4,999",
    "5,000-7,499",
    "7,500-9,999",
    "10,000-14,999",
    "15,000-19,999",
    "20,000-24,999",
    "25,000-29,999",
    "30,000-39,999",
    "40,000-49,999",
    "50,000-59,999",
    "60,000-69,999",
    "70,000-79,999",
    "80,000-89,999",
    "90,000-99,999",
    "100,000-124,999",
    "125,000-149,999",
    "150,000-199,999",
    "200,000-249,999",
    "250,000-299,999",
    "300,000-499,999",
    "$500,000-999,999",
    ">$1,000,000",
]
percentages_cell62 = count_then_return_percent_62(
    responses_df_2022_cell10, question_of_interest_cell62
).sort_index()[responses_in_order_cell62]
percentages_cell62.info()

In [ ]:
### cell 50 ###


def count_then_return_percent_63(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell63 = (
    "What is your current yearly compensation (approximate $USD)?"
)
USA_responses_df_2022_cell63 = responses_df_2022_cell10[
    responses_df_2022_cell10["In which country do you currently reside?"]
    == "United States of America"
]
responses_in_order_cell63 = [
    "$0-999",
    "1,000-1,999",
    "2,000-2,999",
    "3,000-3,999",
    "5,000-7,499",
    "7,500-9,999",
    "10,000-14,999",
    "15,000-19,999",
    "20,000-24,999",
    "25,000-29,999",
    "30,000-39,999",
    "40,000-49,999",
    "50,000-59,999",
    "60,000-69,999",
    "70,000-79,999",
    "80,000-89,999",
    "90,000-99,999",
    "100,000-124,999",
    "125,000-149,999",
    "150,000-199,999",
    "200,000-249,999",
    "250,000-299,999",
    "300,000-499,999",
    "$500,000-999,999",
    ">$1,000,000",
]
percentages_cell63 = (
    count_then_return_percent_63(
        USA_responses_df_2022_cell63, question_of_interest_cell63
    )
    .sort_index()
    .reindex(responses_in_order_cell63, fill_value=0)
)
percentages_cell63.info()

In [ ]:
### cell 51 ###


def count_then_return_percent_64(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell64 = (
    "What is your current yearly compensation (approximate $USD)?"
)
India_responses_df_2022_cell64 = responses_df_2022_cell10[
    responses_df_2022_cell10["In which country do you currently reside?"] == "India"
]
responses_in_order_cell64 = [
    "$0-999",
    "1,000-1,999",
    "2,000-2,999",
    "3,000-3,999",
    "4,000-4,999",
    "5,000-7,499",
    "7,500-9,999",
    "10,000-14,999",
    "15,000-19,999",
    "20,000-24,999",
    "25,000-29,999",
    "30,000-39,999",
    "40,000-49,999",
    "50,000-59,999",
    "60,000-69,999",
    "70,000-79,999",
    "80,000-89,999",
    "90,000-99,999",
    "100,000-124,999",
    "125,000-149,999",
    "150,000-199,999",
    "200,000-249,999",
    "250,000-299,999",
    "300,000-499,999",
    "$500,000-999,999",
    ">$1,000,000",
]
percentages_cell64 = (
    count_then_return_percent_64(
        India_responses_df_2022_cell64, question_of_interest_cell64
    )
    .sort_index()
    .reindex(responses_in_order_cell64, fill_value=0)
)
percentages_cell64.info()

In [ ]:
### cell 52 ###


def count_then_return_percent_65(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


responses_in_order_cell65 = [
    "$0 ($USD)",
    "$1-$99",
    "$100-$999",
    "$1000-$9,999",
    "$10,000-$99,999",
    "$100,000 or more ($USD)",
]
question_of_interest_cell65 = """Approximately how much money have you spent on machine learning and/or cloud computing services at home or at work in the past 5 years (approximate $USD)?\n (approximate $USD)?"""
percentages_cell65 = (
    count_then_return_percent_65(responses_df_2022_cell10, question_of_interest_cell65)
    .sort_index()
    .reindex(responses_in_order_cell65, fill_value=0)
)
percentages_cell65.info()

In [ ]:
### cell 53 ###


def grab_subset_of_data_66(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell66 = "Which of the following cloud computing platforms do you use? (Select all that apply)"
models_df_2022_cell66 = grab_subset_of_data_66(
    responses_df_2022_cell10, question_of_interest_cell66
)
models_df_2022_cell66.info()

In [ ]:
### cell 54 ###


def grab_subset_of_data_67(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_67(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_67(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_67(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_67(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_67(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_67(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_67(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_67(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_67(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_67(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_67(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_67(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_67(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_67(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_67(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_cell67 = "Which of the following cloud computing platforms do you use? (Select all that apply)"
question_of_interest_old_cell67 = (
    "Which of the following cloud computing platforms do you use on a regular basis?"
)
question_of_interest_old_2 = "Which of the following cloud computing services have you used at work or school in the last 5 years?"
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Amazon Web Services (AWS)", "Amazon Web Services (AWS) ", regex=False
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Google Cloud Platform (GCP)", "Google Cloud Platform (GCP) ", regex=False
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Microsoft Azure", "Microsoft Azure ", regex=False
)
responses_df_2021.columns = responses_df_2021.columns.str.replace(
    question_of_interest_old_cell67, question_of_interest_cell67, regex=False
)
responses_df_2020.columns = responses_df_2020.columns.str.replace(
    question_of_interest_old_cell67, question_of_interest_cell67, regex=False
)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    question_of_interest_old_cell67, question_of_interest_cell67, regex=False
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_old_2, question_of_interest_cell67, regex=False
)
(cloud_df_combined, cloud_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(
        question_of_interest_cell67
    )
)
cloud_df_combined_percentages = convert_df_of_counts_to_percentages_67(
    cloud_df_combined, cloud_df_combined_counts
)
cloud_df_combined_percentages_cell67 = cloud_df_combined_percentages.loc[
    :,
    [
        "year",
        "Amazon Web Services (AWS) ",
        "Google Cloud Platform (GCP) ",
        "Microsoft Azure ",
    ],
]
df_cell67 = cloud_df_combined_percentages_cell67.melt(
    id_vars=["year"],
    value_vars=[
        "Amazon Web Services (AWS) ",
        "Google Cloud Platform (GCP) ",
        "Microsoft Azure ",
    ],
)
df_cell67 = df.rename(columns={"variable": ""})
df_cell67.info()

In [ ]:
### cell 55 ###


def count_then_return_percent_68(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell68 = "Of the cloud platforms that you are familiar with, which has the best developer experience (most enjoyable to use)? - Selected Choice"
responses_in_order_cell68 = [
    " Amazon Web Services (AWS) ",
    " Google Cloud Platform (GCP) ",
    " Microsoft Azure ",
    " IBM Cloud / Red Hat ",
    " Oracle Cloud ",
    " VMware Cloud ",
    " SAP Cloud ",
    " Tencent Cloud ",
    " Alibaba Cloud ",
    " Huawei Cloud ",
]
percentages_cell68 = (
    count_then_return_percent_68(responses_df_2022_cell10, question_of_interest_cell68)
    .sort_index()
    .reindex(responses_in_order_cell68, fill_value=0)
    .iloc[::-1]
)
percentages_cell68.info()

In [ ]:
### cell 56 ###


def grab_subset_of_data_69(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell69 = (
    "Do you use any of the following cloud computing products?"
)
compute_df_2022 = grab_subset_of_data_69(
    responses_df_2022_cell10, question_of_interest_cell69
)
compute_df_2022.info()

In [ ]:
### cell 57 ###


def grab_subset_of_data_70(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell70 = "Do you use any of the following data storage products?"
responses_df_2022_cell10.columns = responses_df_2022_cell10.columns.str.replace(
    "(AWS/GCP/Azure customers only)", "for AWS GCP and Azure customers"
)
storage_df_2022 = grab_subset_of_data_70(
    responses_df_2022_cell10, question_of_interest_cell70
)
storage_df_2022.info()

In [ ]:
### cell 58 ###


def grab_subset_of_data_71(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell71 = "Do you use any of the following data products (relational databases, data warehouses, data lakes, or similar)?"
big_data_df_2022 = grab_subset_of_data_71(
    responses_df_2022_cell10, question_of_interest_cell71
)
big_data_df_2022.info()

In [ ]:
### cell 59 ###


def grab_subset_of_data_72(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell72 = (
    "Do you use any of the following business intelligence tools?"
)
bi_df_2022 = grab_subset_of_data_72(
    responses_df_2022_cell10, question_of_interest_cell72
)
bi_df_2022.info()

In [ ]:
### cell 60 ###


def grab_subset_of_data_73(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell73 = "Do you use any of the following managed machine learning products on a regular basis?"
managed_ml_df_2022 = grab_subset_of_data_73(
    responses_df_2022_cell10, question_of_interest_cell73
)
managed_ml_df_2022.info()

In [ ]:
### cell 61 ###


def grab_subset_of_data_74(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell74 = (
    "Do you use any of the following automated machine learning tools?"
)
automl_df_2022 = grab_subset_of_data_74(
    responses_df_2022_cell10, question_of_interest_cell74
)
automl_df_2022.info()

In [ ]:
### cell 62 ###


def grab_subset_of_data_75(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell75 = (
    "Do you use any of the following products to serve your machine learning models?"
)
serving_df_2022 = grab_subset_of_data_75(
    responses_df_2022_cell10, question_of_interest_cell75
)
serving_df_2022.info()

In [ ]:
### cell 63 ###


def grab_subset_of_data_76(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell76 = "Do you use any tools to help monitor your machine learning models and/or experiments?"
tracking_df_2022 = grab_subset_of_data_76(
    responses_df_2022_cell10, question_of_interest_cell76
)
tracking_df_2022.info()

In [ ]:
### cell 64 ###


def grab_subset_of_data_77(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell77 = "Do you use any of the following responsible or ethical AI products in your machine learning practices?"
responsible_df_2022 = grab_subset_of_data_77(
    responses_df_2022_cell10, question_of_interest_cell77
)
responsible_df_2022.info()

In [ ]:
### cell 65 ###


def grab_subset_of_data_78(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell78 = "Do you use any of the following types of specialized hardware when training machine learning models?"
hardware_df_2022 = grab_subset_of_data_78(
    responses_df_2022_cell10, question_of_interest_cell78
)
hardware_df_2022.info()

In [ ]:
### cell 66 ###


def grab_subset_of_data_79(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_2022 = "Do you use any of the following types of specialized hardware when training machine learning models?"
question_of_interest_2019 = (
    "Which types of specialized hardware do you use on a regular basis?"
)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    question_of_interest_2019, question_of_interest_2022, regex=False
)
question_of_interest_2020 = (
    "Which types of specialized hardware do you use on a regular basis?"
)
responses_df_2020.columns = responses_df_2020.columns.str.replace(
    question_of_interest_2020, question_of_interest_2022, regex=False
)
question_of_interest_2021 = (
    "Which types of specialized hardware do you use on a regular basis?"
)
responses_df_2021.columns = responses_df_2021.columns.str.replace(
    question_of_interest_2021, question_of_interest_2022, regex=False
)
question_of_interest_cell79 = question_of_interest_2022
hardware_df_2019 = grab_subset_of_data_79(
    responses_df_2019_cell10, question_of_interest_cell79
)
counts_2019_cell79 = (
    hardware_df_2019[hardware_df_2019.columns[:]].count().sort_values(ascending=True)
)
percentages_2019_cell79 = counts_2019_cell79 * 100 / len(hardware_df_2019)
percentages_2019_cell79 = percentages_2019_cell79[["TPUs"]]
hardware_df_2020 = grab_subset_of_data_79(
    responses_df_2020, question_of_interest_cell79
)
counts_2020_cell79 = (
    hardware_df_2020[hardware_df_2020.columns[:]].count().sort_values(ascending=True)
)
percentages_2020_cell79 = counts_2020_cell79 * 100 / len(hardware_df_2020)
percentages_2020_cell79 = percentages_2020_cell79[["TPUs"]]
hardware_df_2021 = grab_subset_of_data_79(
    responses_df_2021, question_of_interest_cell79
)
counts_2021_cell79 = (
    hardware_df_2021[hardware_df_2021.columns[:]].count().sort_values(ascending=True)
)
percentages_2021_cell79 = counts_2021_cell79 * 100 / len(hardware_df_2021)
percentages_2021_cell79 = percentages_2021_cell79.rename(
    {"Google Cloud TPUs ": "TPUs", "NVIDIA GPUs ": "GPUs"}
)
percentages_2021_cell79 = percentages_2021_cell79[["TPUs"]]
hardware_df_2022_cell79 = grab_subset_of_data_79(
    responses_df_2022_cell10, question_of_interest_cell79
)
counts_2022_cell79 = (
    hardware_df_2022_cell79[hardware_df_2022_cell79.columns[:]]
    .count()
    .sort_values(ascending=True)
)
percentages_2022_cell79 = counts_2022_cell79 * 100 / len(hardware_df_2022_cell79)
percentages_2022_cell79 = percentages_2022_cell79.rename(
    {"TPUs ": "TPUs", "GPUs ": "GPUs"}
)
percentages_2022_cell79 = percentages_2022_cell79[["TPUs"]]
(
    percentages_2019_cell79.info(),
    percentages_2020_cell79.info(),
    percentages_2021_cell79.info(),
    percentages_2022_cell79.info(),
)

In [ ]:
### cell 67 ###


def count_then_return_percent_80(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell80 = (
    "Approximately how many times have you used a TPU (tensor processing unit)?"
)
responses_in_order_cell80 = [
    "Never",
    "Once",
    "2-5 times",
    "6-25 times",
    "More than 25 times",
]
percentages_cell80 = count_then_return_percent_80(
    responses_df_2022_cell10, question_of_interest_cell80
).sort_index()[responses_in_order_cell80]
percentages_cell80.info()

In [ ]:
### cell 68 ###


def count_then_return_percent_81(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_2022_cell81 = (
    "Approximately how many times have you used a TPU (tensor processing unit)?"
)
question_of_interest_2019_cell81 = "Have you ever used a TPU (tensor processing unit)?"
responses_df_2019_cell10[question_of_interest_2019_cell81] = responses_df_2019_cell10[
    question_of_interest_2019_cell81
].str.replace("6-24 times", "6-25 times", regex=False)
responses_df_2019_cell10[question_of_interest_2019_cell81] = responses_df_2019_cell10[
    question_of_interest_2019_cell81
].str.replace("> 25 times", "More than 25 times", regex=False)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    question_of_interest_2019_cell81, question_of_interest_2022_cell81, regex=False
)
question_of_interest_cell81 = question_of_interest_2022_cell81
df_2022 = pd.DataFrame(
    count_then_return_percent_81(
        responses_df_2022_cell10, question_of_interest_cell81
    ).sort_index()
)
df_2021 = pd.DataFrame(
    count_then_return_percent_81(
        responses_df_2021, question_of_interest_cell81
    ).sort_index()
)
df_2020 = pd.DataFrame(
    count_then_return_percent_81(
        responses_df_2020, question_of_interest_cell81
    ).sort_index()
)
df_2019 = pd.DataFrame(
    count_then_return_percent_81(
        responses_df_2019_cell10, question_of_interest_cell81
    ).sort_index()
)
df_2022["year"] = "2022"
df_2021["year"] = "2021"
df_2020["year"] = "2020"
df_2019["year"] = "2019"
df_combined = pd.concat([df_2019, df_2020, df_2021, df_2022])
df_combined[x_axis_title] = df_combined.index
x_axis_title_cell81 = "Frequency of TPU usage"
df_combined.columns = ["percentage", "year", x_axis_title_cell81]
question_of_interest_cell81 = question_of_interest_2022_cell81
df_combined_cell81 = df_combined.sort_values(by=["year", "percentage"], ascending=True)
df_combined_cell81.info()

In [ ]:
### cell 69 ###


def grab_subset_of_data_82(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell82 = (
    "Who/what are your favorite media sources that report on data science topics?"
)
media_df_2022 = grab_subset_of_data_82(
    responses_df_2022_cell10, question_of_interest_cell82
)
media_df_2022.info()

In [ ]:
### cell 70 ###

(
    responses_df_2017.shape[0],
    responses_df_2018_cell10.shape[0],
    responses_df_2019_cell10.shape[0],
    responses_df_2020.shape[0],
    responses_df_2021.shape[0],
    responses_df_2022_cell10.shape[0],
)

In [ ]:
### cell 71 ###

responses_only_duration = responses_df_2022_cell10["Duration (in seconds)"]
responses_only_duration_cell86 = pd.DataFrame(
    pd.to_numeric(responses_only_duration, errors="coerce") / 60
)
responses_only_duration_cell86.columns = ["Duration (in minutes)"]
median = round(responses_only_duration_cell86["Duration (in minutes)"].median(), 0)
print("The median response time was approximately", median, "minutes.")
responses_only_duration_longer_5m = responses_only_duration_cell86[
    responses_only_duration_cell86["Duration (in minutes)"] > 5
]
print(
    "The total number of respondents that took more than 5 minutes was",
    responses_only_duration_longer_5m.shape[0],
    "out of",
    responses_df_2022_cell10.shape[0],
)